# COE Scenario Generation
Generate creative scenarios for error chains using GPT-4o batch API.


In [1]:
import os
import sys
import json
from pathlib import Path
from tokens import openai_key

NOTEBOOK_ROOT = Path("/scratch/jq2uw/MME/instruct_vlm_edit")
os.chdir(NOTEBOOK_ROOT)
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.append(str(NOTEBOOK_ROOT))

from revlm.metrics.utils.e_gen_scenario import COEScenarioGenerator


In [2]:
# Initialize generator
# MODEL_NAME = "Qwen3-VL-8B-Instruct"
# MODEL_NAME = "Qwen3-VL-4B-Instruct"
# MODEL_NAME = "llava-1.5-7b-hf"
MODEL_NAME = "instructblip-vicuna-7b"

# DATASET = "fvqa"  
DATASET = "aokvqa" 

gen = COEScenarioGenerator(
    dataset_name=DATASET,
    model_name=MODEL_NAME,
    openai_key=openai_key
)


COEScenarioGenerator: instructblip-vicuna-7b/aokvqa, 20 batches


## Step 1: Load COE results and prepare batch input


In [3]:
# Load COE prediction results
coe_path = f"results/pred_postedit/baseline/{MODEL_NAME}/{DATASET}/coe_prediction.json"
with open(coe_path, "r") as f:
    coe_results = json.load(f)

print(f"Loaded {len(coe_results)} COE results from {coe_path}")

# Check how many have error chains
with_errors = [r for r in coe_results if gen._get_error_chains(r)]
print(f"Samples with error chains: {len(with_errors)}/{len(coe_results)}")

# Preview a few
for r in with_errors[:3]:
    chains = gen._get_error_chains(r)
    print(f"uid={r['uid']}: {len(chains)} error chains")

# Preview the prompt for one sample
sample = with_errors[1]
chains = gen._get_error_chains(sample)
print(f"Sample has {len(chains)} error chains. Showing first:\n")
print("=== PROMPT ===")
print(gen.format_prompt(chains[0]["chain"]))

Loaded 5954 COE results from results/pred_postedit/baseline/instructblip-vicuna-7b/aokvqa/coe_prediction.json
Samples with error chains: 1950/5954
uid=1: 10 error chains
uid=6: 1 error chains
uid=17: 1 error chains
Sample has 1 error chains. Showing first:

=== PROMPT ===
Given these visual facts:
"The sign mentions a Broadway show. Where would one most likely see the show advertised in the poster is theater."

Generate 3 different creative scenarios where ALL these facts would be visually true.

Requirements:
- Each scenario should be a distinct visual setting
- Use 1-2 short sentences
- Be creative but plausible
- Describe what would be visible in the image

Examples:
Visual facts: "A person is standing on a board. There are waves around."
1. A surfer rides a wave at a tropical beach during sunset. Palm trees line the shore in the background.
2. A wakeboarder is pulled behind a speedboat on a lake. Mountains are visible in the distance.
3. A paddleboarder balances on calm ocean water

In [4]:
# Generate batch request files (uncomment to run)
gen.run_input(coe_results, max_sentences=None)  # or max_sentences=5 to filter


Created 5964 requests in data/coe_gen/instructblip-vicuna-7b/aokvqa/requests/batch.jsonl


## Step 2: Test with ONE batch before firing all


In [5]:
# Check batch files created
batch_files = list(gen.batch_dir.glob("batch_*.jsonl"))
print(f"Batch files: {len(batch_files)}")
for bf in sorted(batch_files)[:5]:
    with open(bf) as f:
        n_lines = len(f.readlines())
    print(f"  {bf.name}: {n_lines} requests")
    
# Preview first request in batch_0
batch_0 = gen.batch_dir / "batch_0.jsonl"
if batch_0.exists():
    with open(batch_0) as f:
        first_req = json.loads(f.readline())
    print(json.dumps(first_req, indent=2)[:1000])  # truncate if too long



Batch files: 20
  batch_0.jsonl: 299 requests
  batch_1.jsonl: 299 requests
  batch_10.jsonl: 299 requests
  batch_11.jsonl: 299 requests
  batch_12.jsonl: 299 requests
{
  "custom_id": "1_[1]",
  "method": "POST",
  "url": "/v1/chat/completions",
  "body": {
    "model": "gpt-4o-mini",
    "messages": [
      {
        "role": "system",
        "content": "You generate creative visual scenarios from given facts."
      },
      {
        "role": "user",
        "content": "Given these visual facts:\n\"A train would not be on the street. What is the man by the bags awaiting is cab.\"\n\nGenerate 3 different creative scenarios where ALL these facts would be visually true.\n\nRequirements:\n- Each scenario should be a distinct visual setting\n- Use 1-2 short sentences\n- Be creative but plausible\n- Describe what would be visible in the image\n\nExamples:\nVisual facts: \"A person is standing on a board. There are waves around.\"\n1. A surfer rides a wave at a tropical beach during sunse

In [6]:
# Submit ONLY batch 0 first (uncomment to run)
gen._run_request_batch(0)


Batch 0: batch_6946c3a2dbf081908964c7dc5fe65b3f


In [7]:
# Get batch 0 results (after it completes)
try:
    results_0 = gen._get_response_batch(0)
    print(f"Got {len(results_0)} results from batch 0")
    
    # Preview first few
    for r in results_0[:3]:
        print(f"\nuid={r['uid']}:")
        for i, s in enumerate(r['scenarios'], 1):
            print(f"  {i}. {s}")
except Exception as e:
    print(f"Error: {e}")


Error: Batch 0 not ready (status: validating)


## Step 3: Fire all batches (after testing batch 0)


In [8]:
# Submit all remaining batches (uncomment to run)
gen.run_request()


Batch 1: batch_6946c3a448688190beddf237c2d4ee44
Batch 2: batch_6946c3a548148190bf29c5f1c89116cc
Batch 3: batch_6946c3a666e88190a140d6be03b942d6
Batch 4: batch_6946c3a7be0c81909b0df2d5a3c275eb
Batch 5: batch_6946c3a85cdc8190a89345c4523b7bf4
Batch 6: batch_6946c3a97dd081908be0573ebddb7a75
Batch 7: batch_6946c3aa46088190bb0a8f9218dbd5db
Batch 8: batch_6946c3abd958819081a44e6a3db40bec
Batch 9: batch_6946c3acda70819095df12f3bc1ba1f8
Batch 10: batch_6946c3ae23f48190bea351e3c92f005b
Batch 11: batch_6946c3af36848190a9408b7820d91228
Batch 12: batch_6946c3b036e481908951b6168ffc918c
Batch 13: batch_6946c3b1b81c8190a0788173dc3fd224
Batch 14: batch_6946c3b2b4388190bf6614782af96677
Batch 15: batch_6946c3b3e67c819099716806c6d38f04
Batch 16: batch_6946c3b502808190845908c297755ed5
Batch 17: batch_6946c3b5a76c8190a6870a178955ee6c
Batch 18: batch_6946c3b8a1808190b9fb451c8745a8d9
Batch 19: batch_6946c3b9a5d081908a49f9b79240893a


In [9]:
# Check status of all submitted batches
for b in range(gen.n_batches):
    meta_path = gen.meta_dir / f"meta_{b}.json"
    if meta_path.exists():
        with open(meta_path) as f:
            meta = json.load(f)
        try:
            job = gen.client.batches.retrieve(meta["job_id"])
            print(f"Batch {b}: {job.status}")
        except Exception as e:
            print(f"Batch {b}: error - {e}")


Batch 0: in_progress
Batch 1: in_progress
Batch 2: in_progress
Batch 3: in_progress
Batch 4: in_progress
Batch 5: in_progress
Batch 6: in_progress
Batch 7: in_progress
Batch 8: in_progress
Batch 9: in_progress
Batch 10: in_progress
Batch 11: in_progress
Batch 12: in_progress
Batch 13: in_progress
Batch 14: in_progress
Batch 15: in_progress
Batch 16: in_progress
Batch 17: in_progress
Batch 18: validating
Batch 19: in_progress


In [10]:
# Resubmit failed batches if needed (uncomment and modify list)
# gen.resubmit_request([0, 1, 2])  # list of batch indices to resubmit


## Step 4: Get all results


In [11]:
# Get all scenarios (after all batches complete)
all_scenarios = gen.get_scenarios()
print(f"Total scenarios: {len(all_scenarios)}")

# Preview results
for r in all_scenarios[:3]:
    print(f"\nuid={r['uid']}, indices={r['indices']}:")
    for i, s in enumerate(r['scenarios'], 1):
        print(f"  {i}. {s}")


Batch 0 error: Batch 0 not ready (status: in_progress)
Batch 1 error: Batch 1 not ready (status: in_progress)
Batch 2 error: Batch 2 not ready (status: in_progress)
Batch 3 error: Batch 3 not ready (status: in_progress)
Batch 4 error: Batch 4 not ready (status: in_progress)
Batch 5 error: Batch 5 not ready (status: in_progress)


Batch 6 error: Batch 6 not ready (status: in_progress)
Batch 7 error: Batch 7 not ready (status: in_progress)
Batch 8 error: Batch 8 not ready (status: in_progress)
Batch 9 error: Batch 9 not ready (status: in_progress)
Batch 10 error: Batch 10 not ready (status: in_progress)
Batch 11 error: Batch 11 not ready (status: in_progress)
Batch 12 error: Batch 12 not ready (status: in_progress)
Batch 13 error: Batch 13 not ready (status: in_progress)
Batch 14 error: Batch 14 not ready (status: in_progress)
Batch 15 error: Batch 15 not ready (status: in_progress)
Batch 16 error: Batch 16 not ready (status: in_progress)
Batch 17 error: Batch 17 not ready (status: in_progress)
Batch 18 error: Batch 18 not ready (status: in_progress)
Batch 19 error: Batch 19 not ready (status: in_progress)
Total scenarios: 0


In [12]:
all_scenarios

[]